In [ ]:
import matplotlib.pyplot as plt
plt.style.use("utils/RESS.mplstyle")

## Figure 5
 
Plot the model for different UQ domains

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns


def plot_sabater2d_full_analysis(
    model_class,
    design_point=(5.0, 5.0),
    uq_domains=("linear", "quad", "non-lin"),
    title=("Linear case $g_1$", "Nonlinear case $g_2$", "Nonlinear case $g_3$"),
    n_grid_x=160,
    n_grid_xi1=220,
    n_grid_xi2=220,
    n_mc_pushforward=50000,
    n_mc_q95_design=6000,
    contour_levels=30,
    figsize_main=(18, 10),
    figsize_q95=(8, 6),
    filename="sabater_domain",
    seed=1234,
    device="cpu",
    dtype=torch.float64,
):
    """
    1) Create a 2x3 plot:
       - first row: stochastic-domain contours M(x_d, xi)
         with the q95 level set M(x_d, xi) = q95
       - second row: push-forward distributions of Y = M(x_d, xi)

       Uses one individual colorbar per first-row subplot.

    2) Create a separate contour plot of the 95% quantile over the design domain:
       q_0.95(x) = Quantile_0.95[M(x, xi)]
       and mark the minimizing design with a star.
    """

    torch.manual_seed(seed)
    np.random.seed(seed)

    fig, axes = plt.subplots(2, len(uq_domains), figsize=figsize_main)
    if len(uq_domains) == 1:
        axes = np.array([[axes[0]], [axes[1]]])

    x_d = torch.tensor(design_point, device=device, dtype=dtype).reshape(1, -1)

    q95_surfaces = {}
    q95_optima = {}

    for j, uq_domain in enumerate(uq_domains):
        model = model_class(uq_domain=uq_domain)

        if uq_domain == "non-lin" and not hasattr(model, "xi1_center"):
            model.xi1_center = 5.0

        # -----------------------------
        # stochastic-space plotting range
        # -----------------------------
        xi1_min = model.xi_dist.dist[0].low
        xi1_max = model.xi_dist.dist[0].high

        xi2_mean = model.xi_dist.dist[1].mean
        xi2_std = model.xi_dist.dist[1].std
        xi2_min = xi2_mean - 4.0 * xi2_std
        xi2_max = xi2_mean + 4.0 * xi2_std

        xi1_grid = torch.linspace(xi1_min, xi1_max, n_grid_xi1, device=device, dtype=dtype)
        xi2_grid = torch.linspace(xi2_min, xi2_max, n_grid_xi2, device=device, dtype=dtype)
        XI1, XI2 = torch.meshgrid(xi1_grid, xi2_grid, indexing="xy")

        xi_eval = torch.stack([XI1.reshape(-1), XI2.reshape(-1)], dim=1)
        x_eval = x_d.repeat(xi_eval.shape[0], 1)

        Y_grid = model.eval(x_eval, xi_eval).reshape(n_grid_xi2, n_grid_xi1).detach().cpu().numpy()

        # -----------------------------
        # MC push-forward at fixed design point
        # -----------------------------
        xi_mc = model.xi_dist.sample(n_mc_pushforward, device=device, dtype=dtype)
        x_mc = x_d.repeat(n_mc_pushforward, 1)
        y_mc = model.eval(x_mc, xi_mc).squeeze(-1).detach().cpu().numpy()
        q95 = np.quantile(y_mc, 0.95)

        # -----------------------------
        # first row: stochastic domain
        # -----------------------------
        ax0 = axes[0, j]
        cf = ax0.contourf(
            XI1.detach().cpu().numpy(),
            XI2.detach().cpu().numpy(),
            Y_grid,
            levels=contour_levels,
            cmap="viridis",
            antialiased = True,
        )
        cf.set_edgecolors("face")

        c95 = ax0.contour(
            XI1.detach().cpu().numpy(),
            XI2.detach().cpu().numpy(),
            Y_grid,
            levels=[q95],
            colors="red",
            linewidths=1.0,
        )
        # ax0.clabel(c95, inline=True, fontsize=9, fmt={q95: "q95"})
        ax0.set_title(f"{title[j]}")
        ax0.set_xlabel(r"$\xi_1$")
        ax0.set_ylabel(r"$\xi_2$")
        # fig.colorbar(cf, ax=ax0, shrink=0.9, pad=0.02, label=r"$M(x_d,\xi)$")

        # -----------------------------
        # second row: push-forward distribution
        # -----------------------------
        ax1 = axes[1, j]
        ax1.hist(
            y_mc,
            bins=100,
            density=True,
            alpha=0.75,
            color="steelblue",
            edgecolor="none",
        )
        ax1.axvline(q95, color="red", linestyle="-", linewidth=1, label="$q_{95}$", ymin=0, ymax=0.5)
        # ax1.set_title(f"{uq_domain}: push-forward distribution")
        ax1.set_xlabel(r"$Y = \mathcal{M}(x,\xi)$")
        ax1.set_ylabel("density")
        # if j == 1: ax1.legend(frameon = False)
        sns.despine(ax=ax1, top=True, right=True)

        # ----------------------------------------
        # Part 2 data: q95 over design space
        # ----------------------------------------
        x1_min, x1_max = model.x_bounds[0]
        x2_min, x2_max = model.x_bounds[1]

        x1_grid = np.linspace(x1_min, x1_max, n_grid_x)
        x2_grid = np.linspace(x2_min, x2_max, n_grid_x)
        X1, X2 = np.meshgrid(x1_grid, x2_grid, indexing="xy")
        Q95 = np.zeros_like(X1)

        xi_q = model.xi_dist.sample(n_mc_q95_design, device=device, dtype=dtype)

        for row in range(n_grid_x):
            for col in range(n_grid_x):
                x_here = torch.tensor(
                    [[X1[row, col], X2[row, col]]],
                    device=device,
                    dtype=dtype,
                ).repeat(n_mc_q95_design, 1)

                y_here = model.eval(x_here, xi_q).squeeze(-1)
                Q95[row, col] = torch.quantile(y_here, 0.95).item()

        min_idx = np.argmin(Q95)
        i_opt, j_opt = np.unravel_index(min_idx, Q95.shape)
        x_opt = (X1[i_opt, j_opt], X2[i_opt, j_opt])
        q95_opt = Q95[i_opt, j_opt]

        q95_surfaces[uq_domain] = (X1, X2, Q95)
        q95_optima[uq_domain] = (x_opt, q95_opt)

    plt.tight_layout()

    plot_folder = "_plots/RESS"
    fig.tight_layout()
    fig.savefig(f'{plot_folder}/{filename}.pdf', format = 'pdf', bbox_inches = 'tight', pad_inches = 0.0)

from models.library import Sabater2d

filename = "Sabater_UQdomain"
figsize = np.array([300,200])/72.27

plot_sabater2d_full_analysis(
    model_class=Sabater2d,
    design_point=(4.0, 4.0),
    uq_domains=("linear", "quad", "nonlin"),
    filename = filename,
    title = ("Linear case $g_1$", "Nonlinear case $g_2$", "Nonlinear case $g_3$"),
    n_grid_x=50,
    n_grid_xi1=50,
    n_grid_xi2=50,
    n_mc_pushforward=100000,
    n_mc_q95_design=4000,
    figsize_main=figsize
)


## Figure 6

In [ ]:
from matplotlib.patches import FancyArrowPatch
def add_double_arrow(ax, start_x, end_x, y_position, label, shift_text_y = 0.02):
    # Create a double arrow to represent the interval
    arrow_length = 0.2  # Adjust this for arrow length
    arrow = FancyArrowPatch((start_x, y_position), (end_x, y_position),
                             arrowstyle='<|-|>', color='black', lw=1,
                             mutation_scale=10, shrinkA= 0.0, shrinkB= 0.0)
    ax.add_patch(arrow)
    
    # Add the label in the middle of the arrow
    ax.text((start_x + end_x) / 2, y_position + shift_text_y, label, ha='center', va='bottom', fontsize=8)

    return ax

def add_iteration_arrows(ax, model_calls_cumsum, n_EI, n_EM, ypos = None, ymin = None, shift_text_y = 0.04):
    ylim = ax.get_ylim()
    delta_y = ylim[1]-ylim[0]
    shift = 0.075

    if ypos is None:
        ypos = ylim[0]-shift*delta_y

    ax = add_double_arrow(ax = ax, start_x=model_calls_cumsum[0], end_x=model_calls_cumsum[n_EI], y_position=ypos, label = "EI", shift_text_y = shift_text_y)
    ax = add_double_arrow(ax = ax, start_x=model_calls_cumsum[n_EI], end_x=model_calls_cumsum[n_EI+n_EM], y_position=ypos, label = "EM", shift_text_y = shift_text_y)
    ax = add_double_arrow(ax = ax, start_x=model_calls_cumsum[n_EI+n_EM], end_x=model_calls_cumsum[-1], y_position=ypos, label = "TR", shift_text_y = shift_text_y)
    
    if ymin is None:
        ax.set_ylim([ylim[0]-1.5*shift*delta_y, ylim[1]])
    else:
        ax.set_ylim([ymin, ylim[1]])
    
    return ax

In [ ]:
import os, torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from notebooks.helper import get_batch_sim
ccycle = plt.rcParams['axes.prop_cycle'].by_key()['color']

directory = '_experiments/RESS/sabater_batch_run_paper'
data_list = ["linear_uq", "quad_uq", "nonlin_uq"]

ccycle = plt.rcParams['axes.prop_cycle'].by_key()['color']

figsize = np.array([300,200])/72.27

fig, ax = plt.subplots(2, 3, figsize=figsize, sharex=True, sharey='row')

title_text = ["Case $g_1$", "Case $g_2$", "Case $g_3$"]

eps = 1e-12


for i, data in enumerate(data_list):
    best_q, best_x, model_calls_cumsum, n_EI, n_EM, n_TR, ref_x, ref_q = get_batch_sim(directory, data)

    best_q_rel = (torch.abs(best_q - ref_q) / torch.abs(ref_q)).clamp_min(eps)

    true_norm = torch.linalg.norm(ref_x).clamp_min(eps)
    best_x_rel = (torch.linalg.norm(best_x - ref_x, dim=-1) / true_norm).clamp_min(eps)

    q_med = torch.quantile(best_q_rel, 0.5, dim=1, interpolation="nearest")
    q_lo  = torch.quantile(best_q_rel, 0.1, dim=1, interpolation="nearest")
    q_hi  = torch.quantile(best_q_rel, 0.9, dim=1, interpolation="nearest")

    x_med = torch.quantile(best_x_rel, 0.5, dim=1, interpolation="nearest")
    x_lo  = torch.quantile(best_x_rel, 0.1, dim=1, interpolation="nearest")
    x_hi  = torch.quantile(best_x_rel, 0.9, dim=1, interpolation="nearest")

    x_axis = model_calls_cumsum.cpu().numpy()

    # top row: q
    ax[0, i].set_title(title_text[i])
    ax[0, i].semilogy(x_axis, best_q_rel.cpu().numpy(), color='0.85', linewidth=1, alpha = 0.9, zorder = 1)
    ax[0, i].semilogy(x_axis, q_med.cpu().numpy(), color=ccycle[0], linewidth=2, linestyle='--', zorder = 100)
    ax[0, i].fill_between(x_axis, q_lo.cpu().numpy(), q_hi.cpu().numpy(), color=ccycle[0], alpha=0.2, zorder = 50)
    ax[0, i].grid(True, alpha=0.3)
    ax[0, i].set_ylim(1e-3, 1e0)

    # bottom row: x
    ax[1, i].semilogy(x_axis, best_x_rel.cpu().numpy(), color='0.85', linewidth=1, alpha = 0.9, zorder = 1)
    ax[1, i].semilogy(x_axis, x_med.cpu().numpy(), color=ccycle[0], linewidth=2, linestyle='--', zorder = 100)
    ax[1, i].fill_between(x_axis, x_lo.cpu().numpy(), x_hi.cpu().numpy(), color=ccycle[0], alpha=0.2, zorder = 50)
    ax[1, i].grid(True, alpha=0.3)
    ax[1, i].set_ylim(1e-3, 1e0)

# only left column keeps y tick labels
for j in range(1, 3):
    ax[0, j].tick_params(labelleft=False)
    ax[1, j].tick_params(labelleft=False)

# only bottom row keeps x tick labels
for j in range(3):
    ax[0, j].tick_params(labelbottom=False)

# axis labels only where wanted
ax[0, 0].set_ylabel(r"$ \epsilon_{q, \mathrm{rel}}$")
ax[1, 0].set_ylabel(r"$ \epsilon_{x, \mathrm{rel}}$")

for j in range(3):
    ax[1, j].set_xlabel('model calls')

# despine top/right
for a in ax.ravel():
    sns.despine(ax=a)

fig.tight_layout()

plot_folder = "_plots/RESS"
file_name = "sabater_results_paper"
fig.tight_layout()
fig.savefig(f'{plot_folder}/{file_name}.pdf', format = 'pdf', bbox_inches = 'tight', pad_inches = 0.0)

## Figure 17 

MLRO for Sabater test function + g_3 with increased DOE

In [ ]:
import os, torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from notebooks.helper import get_batch_sim

ccycle = plt.rcParams['axes.prop_cycle'].by_key()['color']

# directory = '_experiments/Sabater2d/batch_run_paper_nonlinUQ'
directory = '_experiments/RESS/sabater_batch_run_paper_g3_increased_doe'
data_list = ["nonlin_uq_inc_doeInfill"]
title_text = ["Case $g_1$"]

figsize = np.array([250,100])/72.27
# figsize = np.array([450,200])/72.27
fig, ax = plt.subplots(1, 2, figsize=figsize)

eps = 1e-12

for i, data in enumerate(data_list):
    best_q, best_x, model_calls_cumsum, n_EI, n_EM, n_TR, ref_x, ref_q = get_batch_sim(directory, data)

    best_q_rel = (torch.abs(best_q - ref_q) / torch.abs(ref_q)).clamp_min(eps)

    true_norm = torch.linalg.norm(ref_x).clamp_min(eps)
    best_x_rel = (torch.linalg.norm(best_x - ref_x, dim=-1) / true_norm).clamp_min(eps)

    q_med_adapt = torch.quantile(best_q_rel, 0.5, dim=1, interpolation="nearest")
    q_lo_adapt  = torch.quantile(best_q_rel, 0.1, dim=1, interpolation="nearest")
    q_hi_adapt  = torch.quantile(best_q_rel, 0.9, dim=1, interpolation="nearest")

    x_med_adapt = torch.quantile(best_x_rel, 0.5, dim=1, interpolation="nearest")
    x_lo_adapt  = torch.quantile(best_x_rel, 0.1, dim=1, interpolation="nearest")
    x_hi_adapt  = torch.quantile(best_x_rel, 0.9, dim=1, interpolation="nearest")

    x_axis_adapt = model_calls_cumsum.cpu().numpy()

    ax[0].semilogy(x_axis_adapt, q_med_adapt.cpu().numpy(), color=ccycle[1], linewidth=2, linestyle='--', zorder = 100, label = "Increased DOE")
    ax[0].fill_between(x_axis_adapt, q_lo_adapt.cpu().numpy(), q_hi_adapt.cpu().numpy(), color=ccycle[1], alpha=0.2, zorder = 50)
    ax[0].semilogy(x_axis, q_med.cpu().numpy(), color=ccycle[0], linewidth=2, linestyle='--', zorder = 100, label = "Nominal DOE")
    ax[0].fill_between(x_axis, q_lo.cpu().numpy(), q_hi.cpu().numpy(), color=ccycle[0], alpha=0.2, zorder = 50)
    ax[0].grid(True, alpha=0.3)
    ax[0].set_ylim(1e-3, 1e0)
    ax[0].axhline(0.01, linestyle = "--", color='0.5', linewidth = 1.5)

    ax[1].semilogy(x_axis_adapt, x_med_adapt.cpu().numpy(), color=ccycle[1], linewidth=2, linestyle='--', zorder = 100)
    ax[1].fill_between(x_axis_adapt, x_lo_adapt.cpu().numpy(), x_hi_adapt.cpu().numpy(), color=ccycle[1], alpha=0.2, zorder = 50)
    ax[1].semilogy(x_axis, x_med.cpu().numpy(), color=ccycle[0], linewidth=2, linestyle='--', zorder = 100)
    ax[1].fill_between(x_axis, x_lo.cpu().numpy(), x_hi.cpu().numpy(), color=ccycle[0], alpha=0.2, zorder = 50)
    ax[1].grid(True, alpha=0.3)
    ax[1].set_ylim(1e-3, 1e0)
    ax[1].axhline(0.01, linestyle = "--", color='0.5', linewidth = 1.5)

ax[0].set_ylabel(r"$ \epsilon_{q, \mathrm{rel}}$")
ax[1].set_ylabel(r"$ \epsilon_{x, \mathrm{rel}}$")

ax[0].set_xlabel("model calls")
ax[1].set_xlabel("model calls")

for a in ax:
    sns.despine(ax=a)

handles, labels = ax[0].get_legend_handles_labels()

fig.legend(
    handles,
    labels,
    loc="upper center",
    bbox_to_anchor=(0.5, 1.15),
    ncol=2,
    frameon=False,
)

fig.text(
    0.035, 0.06, r"\textsf{\textbf{(a)}}",
    ha="left",
    va="bottom",
    fontsize=8,
    color="black",
)

fig.text(
    0.52, 0.06, r"\textsf{\textbf{(b)}}",
    ha="left",
    va="bottom",
    fontsize=8,
    color="black",
)

fig.tight_layout()
plt.show()

plot_folder = "_plots/RESS"
file_name = "sabater_results_paper_update"
fig.tight_layout()
fig.savefig(f'{plot_folder}/{file_name}.pdf', format = 'pdf', bbox_inches = 'tight', pad_inches = 0.0)


## Figure 7 

MLRO evolution for Branin-Hoo and Hartmann-6 test function

In [ ]:
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from notebooks.helper import get_batch_sim


ccycle = plt.rcParams["axes.prop_cycle"].by_key()["color"]

eps = 1e-12

# One configuration per column
experiments = [
    {
        "directory": "_experiments/RESS/branin_batch_run_paper",
        "data": "nonlin_uq",
        "title": "BraninHoo",
    },
    {
        "directory": "_experiments/RESS/hartmann_batch_run",
        "data": "ro",
        "title": "Hartmann6",
    },
]

# Share the x-axis within each column
fig, ax = plt.subplots(
    2,
    2,
    figsize=np.array([230, 200]) / 72.27,
    sharex="col",
)

for col, experiment in enumerate(experiments):
    best_q, best_x, model_calls_cumsum, n_EI, n_EM, n_TR, ref_x, ref_q = (
        get_batch_sim(
            experiment["directory"],
            experiment["data"],
        )
    )

    # Relative q error
    best_q_rel = (
        torch.abs(best_q - ref_q) / torch.abs(ref_q)
    ).clamp_min(eps)

    # Relative x error
    true_norm = torch.linalg.norm(ref_x).clamp_min(eps)

    best_x_rel = (
        torch.linalg.norm(best_x - ref_x, dim=-1) / true_norm
    ).clamp_min(eps)

    # Quantiles across runs
    q_med = torch.quantile(best_q_rel, 0.5, dim=1, interpolation="nearest")
    q_lo  = torch.quantile(best_q_rel, 0.1, dim=1, interpolation="nearest")
    q_hi  = torch.quantile(best_q_rel, 0.9, dim=1, interpolation="nearest")

    x_med = torch.quantile(best_x_rel, 0.5, dim=1, interpolation="nearest")
    x_lo  = torch.quantile(best_x_rel, 0.1, dim=1, interpolation="nearest")
    x_hi  = torch.quantile(best_x_rel, 0.9, dim=1, interpolation="nearest")

    x_axis = model_calls_cumsum.cpu().numpy()
    q_values = best_q_rel.cpu().numpy()
    x_values = best_x_rel.cpu().numpy()

    q_color = ccycle[col+1]

    # ------------------------------------------------------------------
    # First row: q convergence
    # ------------------------------------------------------------------
    q_ax = ax[0, col]

    q_ax.semilogy(
        x_axis,
        q_values,
        color="0.85",
        linewidth=1,
        alpha=0.5,
        zorder = 1
    )

    q_ax.semilogy(
        x_axis,
        q_med.cpu().numpy(),
        color=q_color,
        linewidth=2,
        linestyle="--",
        zorder = 100
    )

    q_ax.fill_between(
        x_axis,
        q_lo.cpu().numpy(),
        q_hi.cpu().numpy(),
        color=q_color,
        alpha=0.2,
        zorder = 50
    )

    q_ax.set_ylim(1e-4, 1e0)
    q_ax.set_yticks([1e-4, 1e-3, 1e-2, 1e-1, 1e0])
    q_ax.set_title(experiment["title"])

    # Hide x-axis tick labels in the first row
    q_ax.tick_params(
        axis="x",
        which="both",
        labelbottom=False,
    )

    # ------------------------------------------------------------------
    # Second row: x convergence
    # ------------------------------------------------------------------
    x_ax = ax[1, col]

    x_ax.semilogy(
        x_axis,
        x_values,
        color="0.85",
        linewidth=1,
        alpha=0.5,
        zorder = 1
    )

    x_ax.semilogy(
        x_axis,
        x_med.cpu().numpy(),
        color=q_color,
        linewidth=2,
        linestyle="--",
        zorder = 100
    )

    x_ax.fill_between(
        x_axis,
        x_lo.cpu().numpy(),
        x_hi.cpu().numpy(),
        color=q_color,
        alpha=0.2,
        zorder = 50
    )

    x_ax.set_ylim(1e-3, 1e0)

# Axis labels
ax[0, 0].set_ylabel(r"$ \epsilon_{q, \mathrm{rel}}$")
ax[1, 0].set_ylabel(r"$ \epsilon_{x, \mathrm{rel}}$")

ax[1, 0].set_xlabel("model calls")
ax[1, 1].set_xlabel("model calls")

for row in range(2):
    ax[row, 1].tick_params(
        axis="y",
        which="both",
        left=True,       # hide tick marks
        labelleft=False,  # hide tick labels
    )

# Grid and styling
for row in range(2):
    for col in range(2):
        ax[row, col].grid(True, alpha=0.3)
        sns.despine(ax=ax[row, col])

fig.tight_layout()

# Save the combined figure
plot_folder = os.path.join("_plots", "RESS")
os.makedirs(plot_folder, exist_ok=True)

file_name = "branin_hartmann_convergence.pdf"
file_path = os.path.join(plot_folder, file_name)

fig.savefig(
    file_path,
    format="pdf",
    bbox_inches="tight",
    pad_inches=0.0,
)

plt.show()